# Linear Regression Notebook

> Hands-on Build It and Exercises.

## Build It

### Step 1: Generate sample data

In [ ]:
```python

import random

import math

random.seed(42)

TRUE_W = 3.0

TRUE_B = 7.0

N_SAMPLES = 100

X = [random.uniform(0, 10) for _ in range(N_SAMPLES)]

y = [TRUE_W * x + TRUE_B + random.gauss(0, 2.0) for x in X]

print(f"Generated {N_SAMPLES} samples")

print(f"True relationship: y = {TRUE_W}x + {TRUE_B} (+ noise)")

print(f"First 5 points: {[(round(X[i], 2), round(y[i], 2)) for i in range(5)]}")

In [ ]:
```

### Step 2: Linear regression from scratch with gradient descent

In [ ]:
```python

class LinearRegression:

    def __init__(self, learning_rate=0.01):

        self.w = 0.0

        self.b = 0.0

        self.lr = learning_rate

        self.cost_history = []

    def predict(self, X):

        return [self.w * x + self.b for x in X]

    def compute_cost(self, X, y):

        predictions = self.predict(X)

        n = len(y)

        cost = sum((pred - actual) ** 2 for pred, actual in zip(predictions, y)) / n

        return cost

    def compute_gradients(self, X, y):

        predictions = self.predict(X)

        n = len(y)

        dw = (2 / n) * sum((pred - actual) * x for pred, actual, x in zip(predictions, y, X))

        db = (2 / n) * sum(pred - actual for pred, actual in zip(predictions, y))

        return dw, db

    def fit(self, X, y, epochs=1000, print_every=200):

        for epoch in range(epochs):

            dw, db = self.compute_gradients(X, y)

            self.w -= self.lr * dw

            self.b -= self.lr * db

            cost = self.compute_cost(X, y)

            self.cost_history.append(cost)

            if epoch % print_every == 0:

                print(f"  Epoch {epoch:4d} | Cost: {cost:.4f} | w: {self.w:.4f} | b: {self.b:.4f}")

        return self

    def r_squared(self, X, y):

        predictions = self.predict(X)

        y_mean = sum(y) / len(y)

        ss_res = sum((actual - pred) ** 2 for actual, pred in zip(y, predictions))

        ss_tot = sum((actual - y_mean) ** 2 for actual in y)

        return 1 - (ss_res / ss_tot)

print("=== Training Linear Regression (Gradient Descent) ===")

model = LinearRegression(learning_rate=0.005)

model.fit(X, y, epochs=1000, print_every=200)

print(f"\nLearned: y = {model.w:.4f}x + {model.b:.4f}")

print(f"True:    y = {TRUE_W}x + {TRUE_B}")

print(f"R-squared: {model.r_squared(X, y):.4f}")

In [ ]:
```

### Try It: break it, then fix it

Reading working code is not the same as understanding it. The two cells below

each have a problem. Predict what happens *before* you run them — then run, see

the failure, and fix it. The failure is the lesson.

**A. The learning rate is too large.**

Before running: the model above used `learning_rate=0.005` and converged. This

one uses `0.5` — 100x larger. Write down your prediction: does it converge

faster, converge to the same answer, or something else? Then run it.

In [ ]:
```python

# Same model as Step 2, but learning_rate=0.5 instead of 0.005.

model = LinearRegression(learning_rate=0.5)

model.fit(X, y, epochs=30, print_every=5)

print(f"\nLearned: y = {model.w:.4g}x + {model.b:.4g}")

print(f"True:    y = {TRUE_W}x + {TRUE_B}")

In [ ]:
```

You should see the cost *grow* every step instead of shrinking — `5e2`, then

`5e17`, then `4e32` — and `w` flip sign on each epoch. The steps are so large

that gradient descent overshoots the minimum and bounces further out each time.

This is divergence, and it is the single most common reason "my model won't

train." **Fix it:** change `0.5` back to a small value (try `0.01`, `0.005`,

`0.001`) and re-run until the cost shrinks toward zero. There is no single right

answer — that is the point. The learning rate is a dial you tune by watching the

cost.

**B. Fill in the missing gradient.**

The `compute_gradients` method below is blank. This is the heart of the whole

algorithm — the partial derivatives of MSE with respect to `w` and `b`. Derive

them and fill them in. The self-check at the bottom compares your formula

against a numerical gradient and tells you if you got it right — no need to

trust the lesson, the math checks itself.

In [ ]:
```python

class LinearRegressionGap(LinearRegression):

    def compute_gradients(self, X, y):

        predictions = self.predict(X)

        n = len(y)

        # TODO: replace these two lines.

        # dw is the partial derivative of MSE w.r.t. w; db w.r.t. b.

        # Hint: MSE = (1/n) * sum((w*x + b - y)^2). Differentiate, and the

        # factor of 2 from the square comes out front.

        dw = 0.0

        db = 0.0

        return dw, db

# --- self-check: do not edit below this line ---

m = LinearRegressionGap()

m.w, m.b = 1.5, 2.0          # arbitrary point to test the gradient at

dw, db = m.compute_gradients(X, y)

def _cost_at(w, b):

    return sum((w * xi + b - a) ** 2 for xi, a in zip(X, y)) / len(y)

h = 1e-4

num_dw = (_cost_at(1.5 + h, 2.0) - _cost_at(1.5 - h, 2.0)) / (2 * h)

num_db = (_cost_at(1.5, 2.0 + h) - _cost_at(1.5, 2.0 - h)) / (2 * h)

if abs(dw - num_dw) < 1e-2 and abs(db - num_db) < 1e-2:

    print("PASS — your gradient matches the numerical check.")

    print(f"  your dw={dw:.3f}, db={db:.3f}")

else:

    print("Not yet. Your gradient does not match the numerical slope.")

    print(f"  you got    dw={dw:.3f}, db={db:.3f}")

    print(f"  should be  dw={num_dw:.3f}, db={num_db:.3f}")

    print("  Re-derive d/dw and d/db of the mean squared error.")

In [ ]:
```

When it prints `PASS`, you have implemented gradient descent's core yourself —

not read it, derived it. The solution is the `compute_gradients` you saw in

Step 2; only check it after you have tried.

### Step 3: Normal equation (closed-form solution)

In [ ]:
```python

class LinearRegressionNormal:

    def __init__(self):

        self.w = 0.0

        self.b = 0.0

    def fit(self, X, y):

        n = len(X)

        x_mean = sum(X) / n

        y_mean = sum(y) / n

        numerator = sum((X[i] - x_mean) * (y[i] - y_mean) for i in range(n))

        denominator = sum((X[i] - x_mean) ** 2 for i in range(n))

        self.w = numerator / denominator

        self.b = y_mean - self.w * x_mean

        return self

    def predict(self, X):

        return [self.w * x + self.b for x in X]

    def r_squared(self, X, y):

        predictions = self.predict(X)

        y_mean = sum(y) / len(y)

        ss_res = sum((actual - pred) ** 2 for actual, pred in zip(y, predictions))

        ss_tot = sum((actual - y_mean) ** 2 for actual in y)

        return 1 - (ss_res / ss_tot)

print("\n=== Normal Equation (Closed-Form) ===")

model_normal = LinearRegressionNormal()

model_normal.fit(X, y)

print(f"Learned: y = {model_normal.w:.4f}x + {model_normal.b:.4f}")

print(f"R-squared: {model_normal.r_squared(X, y):.4f}")

In [ ]:
```

### Step 4: Multiple linear regression

In [ ]:
```python

class MultipleLinearRegression:

    def __init__(self, n_features, learning_rate=0.01):

        self.weights = [0.0] * n_features

        self.bias = 0.0

        self.lr = learning_rate

        self.cost_history = []

    def predict_single(self, x):

        return sum(w * xi for w, xi in zip(self.weights, x)) + self.bias

    def predict(self, X):

        return [self.predict_single(x) for x in X]

    def compute_cost(self, X, y):

        predictions = self.predict(X)

        n = len(y)

        return sum((pred - actual) ** 2 for pred, actual in zip(predictions, y)) / n

    def fit(self, X, y, epochs=1000, print_every=200):

        n = len(y)

        n_features = len(X[0])

        for epoch in range(epochs):

            predictions = self.predict(X)

            errors = [pred - actual for pred, actual in zip(predictions, y)]

            for j in range(n_features):

                grad = (2 / n) * sum(errors[i] * X[i][j] for i in range(n))

                self.weights[j] -= self.lr * grad

            grad_b = (2 / n) * sum(errors)

            self.bias -= self.lr * grad_b

            cost = self.compute_cost(X, y)

            self.cost_history.append(cost)

            if epoch % print_every == 0:

                print(f"  Epoch {epoch:4d} | Cost: {cost:.4f}")

        return self

    def r_squared(self, X, y):

        predictions = self.predict(X)

        y_mean = sum(y) / len(y)

        ss_res = sum((actual - pred) ** 2 for actual, pred in zip(y, predictions))

        ss_tot = sum((actual - y_mean) ** 2 for actual in y)

        return 1 - (ss_res / ss_tot)

random.seed(42)

N = 100

X_multi = []

y_multi = []

for _ in range(N):

    size = random.uniform(500, 3000)

    bedrooms = random.randint(1, 5)

    age = random.uniform(0, 50)

    price = 50 * size + 10000 * bedrooms - 1000 * age + 50000 + random.gauss(0, 20000)

    X_multi.append([size, bedrooms, age])

    y_multi.append(price)

def standardize(X):

    n_features = len(X[0])

    means = [sum(X[i][j] for i in range(len(X))) / len(X) for j in range(n_features)]

    stds = []

    for j in range(n_features):

        variance = sum((X[i][j] - means[j]) ** 2 for i in range(len(X))) / len(X)

        stds.append(variance ** 0.5)

    X_scaled = []

    for i in range(len(X)):

        row = [(X[i][j] - means[j]) / stds[j] if stds[j] > 0 else 0 for j in range(n_features)]

        X_scaled.append(row)

    return X_scaled, means, stds

y_mean_val = sum(y_multi) / len(y_multi)

y_std_val = (sum((yi - y_mean_val) ** 2 for yi in y_multi) / len(y_multi)) ** 0.5

y_scaled = [(yi - y_mean_val) / y_std_val for yi in y_multi]

X_scaled, x_means, x_stds = standardize(X_multi)

print("\n=== Multiple Linear Regression (3 features) ===")

print("Features: house size, bedrooms, age")

multi_model = MultipleLinearRegression(n_features=3, learning_rate=0.01)

multi_model.fit(X_scaled, y_scaled, epochs=1000, print_every=200)

print(f"\nWeights (standardized): {[round(w, 4) for w in multi_model.weights]}")

print(f"Bias (standardized): {multi_model.bias:.4f}")

print(f"R-squared: {multi_model.r_squared(X_scaled, y_scaled):.4f}")

In [ ]:
```

### Step 5: Polynomial regression

In [ ]:
```python

class PolynomialRegression:

    def __init__(self, degree, learning_rate=0.01):

        self.degree = degree

        self.weights = [0.0] * degree

        self.bias = 0.0

        self.lr = learning_rate

    def make_features(self, X):

        return [[x ** (d + 1) for d in range(self.degree)] for x in X]

    def predict(self, X):

        features = self.make_features(X)

        return [sum(w * f for w, f in zip(self.weights, row)) + self.bias for row in features]

    def fit(self, X, y, epochs=1000, print_every=200):

        features = self.make_features(X)

        n = len(y)

        for epoch in range(epochs):

            predictions = [sum(w * f for w, f in zip(self.weights, row)) + self.bias for row in features]

            errors = [pred - actual for pred, actual in zip(predictions, y)]

            for j in range(self.degree):

                grad = (2 / n) * sum(errors[i] * features[i][j] for i in range(n))

                self.weights[j] -= self.lr * grad

            grad_b = (2 / n) * sum(errors)

            self.bias -= self.lr * grad_b

            if epoch % print_every == 0:

                cost = sum(e ** 2 for e in errors) / n

                print(f"  Epoch {epoch:4d} | Cost: {cost:.6f}")

        return self

    def r_squared(self, X, y):

        predictions = self.predict(X)

        y_mean = sum(y) / len(y)

        ss_res = sum((actual - pred) ** 2 for actual, pred in zip(y, predictions))

        ss_tot = sum((actual - y_mean) ** 2 for actual in y)

        return 1 - (ss_res / ss_tot)

random.seed(42)

X_poly = [x / 10.0 for x in range(0, 50)]

y_poly = [0.5 * x ** 2 - 2 * x + 3 + random.gauss(0, 1.0) for x in X_poly]

x_max = max(abs(x) for x in X_poly)

X_poly_norm = [x / x_max for x in X_poly]

y_poly_mean = sum(y_poly) / len(y_poly)

y_poly_std = (sum((yi - y_poly_mean) ** 2 for yi in y_poly) / len(y_poly)) ** 0.5

y_poly_norm = [(yi - y_poly_mean) / y_poly_std for yi in y_poly]

print("\n=== Polynomial Regression (degree 2 vs degree 5) ===")

print("True relationship: y = 0.5x^2 - 2x + 3")

print("\nDegree 2:")

poly2 = PolynomialRegression(degree=2, learning_rate=0.1)

poly2.fit(X_poly_norm, y_poly_norm, epochs=2000, print_every=500)

print(f"  R-squared: {poly2.r_squared(X_poly_norm, y_poly_norm):.4f}")

print("\nDegree 5:")

poly5 = PolynomialRegression(degree=5, learning_rate=0.1)

poly5.fit(X_poly_norm, y_poly_norm, epochs=2000, print_every=500)

print(f"  R-squared: {poly5.r_squared(X_poly_norm, y_poly_norm):.4f}")

print("\nDegree 2 fits the true curve well. Degree 5 fits training data slightly better")

print("but risks overfitting on new data.")

In [ ]:
```

### Step 6: Ridge regression (L2 regularization)

In [ ]:
```python

class RidgeRegression:

    def __init__(self, n_features, learning_rate=0.01, alpha=1.0):

        self.weights = [0.0] * n_features

        self.bias = 0.0

        self.lr = learning_rate

        self.alpha = alpha

    def predict_single(self, x):

        return sum(w * xi for w, xi in zip(self.weights, x)) + self.bias

    def predict(self, X):

        return [self.predict_single(x) for x in X]

    def fit(self, X, y, epochs=1000, print_every=200):

        n = len(y)

        n_features = len(X[0])

        for epoch in range(epochs):

            predictions = self.predict(X)

            errors = [pred - actual for pred, actual in zip(predictions, y)]

            mse = sum(e ** 2 for e in errors) / n

            reg_term = self.alpha * sum(w ** 2 for w in self.weights)

            cost = mse + reg_term

            for j in range(n_features):

                grad = (2 / n) * sum(errors[i] * X[i][j] for i in range(n))

                grad += 2 * self.alpha * self.weights[j]

                self.weights[j] -= self.lr * grad

            grad_b = (2 / n) * sum(errors)

            self.bias -= self.lr * grad_b

            if epoch % print_every == 0:

                print(f"  Epoch {epoch:4d} | Cost: {cost:.4f} | L2 penalty: {reg_term:.4f}")

        return self

print("\n=== Ridge Regression (L2 Regularization) ===")

print("Same data as multiple regression, with alpha=0.1")

ridge = RidgeRegression(n_features=3, learning_rate=0.01, alpha=0.1)

ridge.fit(X_scaled, y_scaled, epochs=1000, print_every=200)

print(f"\nRidge weights: {[round(w, 4) for w in ridge.weights]}")

print(f"Plain weights: {[round(w, 4) for w in multi_model.weights]}")

print("Ridge weights are smaller (shrunk toward zero) due to the L2 penalty.")

In [ ]:
```

## Exercises

In [ ]:
1. Implement batch gradient descent, stochastic gradient descent (SGD), and mini-batch gradient descent. Compare convergence speed on the same dataset. Which converges fastest? Which has the smoothest cost curve?
2. Generate data from a cubic function (y = ax^3 + bx^2 + cx + d + noise). Fit polynomials of degree 1, 3, and 10. Compare training R^2 and test R^2. At what degree does overfitting become obvious?
3. Implement Lasso regression (L1 regularization: penalty = alpha * sum(|w_i|)). Train on the multi-feature housing data. Compare which weights go to zero vs Ridge. Why does L1 produce sparse solutions while L2 does not?